# Overview 
This notebook will walk you through the different approaches that can be used to create a robust fraud detection classifier.
It starts with the simple Decision Tree model, then the Random Forest model and finally a neural network one.   

**PS:** *In order to get the same results, make sure to switch off GPU*

### Dataset
The dataset is composed of 284,807 transactions made by credit cards during two days in September 2013 by European cardholders.
It contains 492 frauds out of the 284,807 (0.172% of all transactions) which makes it a highly unbalanced dataset.
For more info on the dataset, please check its [Kaggle page](https://www.kaggle.com/mlg-ulb/creditcardfraud)

> # Libraries

Let's import the libraries that we will need.
We will be mainly using:
* `pandas` for data handling
* `sklearn` for DecisionTree and RandomForest
* `tensoforflow.keras` for the neural network model
Along with many others

In [ ]:
# This is importatnt to reproduce the same results
from numpy.random import seed
seed(1)
from tensorflow import set_random_seed
set_random_seed(2)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, roc_curve, auc, precision_recall_curve

import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam

# Useful Functions
Functions that we will be using in different sections below

In [ ]:
def get_features_to_keep(data, correlation_threshold = 0):
  # This function returns the list of features to keep.
  # This list contains any features whose correlation with Class field is strictly greater than correlation_threshold
  # If correlation_threshold is None, we will be keeping all features.
  if correlation_threshold is not None:
    list_of_features = corr.iloc[:, data.columns == 'Class']
    list_of_features = list_of_features[list_of_features['Class'] > correlation_threshold].index.tolist()
  else:
    list_of_features = []
  
  return list_of_features

def get_intersections(c1, c2):
    # This function checks the intersection between 2 curves c1 and c2
    # First it checks if there is an exact match of values.
    # If not, it will check where there was a change of sign for c1 - c2
    intersections = np.argwhere(c1 == c2).flatten()
    if len(intersections) == 0:
        intersections = np.argwhere(np.diff(np.sign(c1 - c2))).flatten()
    
    return intersections

# Data Manipulation

* Read the data from the csv file using `pandas.read_csv`
* Print the data dimensions which are: `(284807, 31)` i.e. `284807` samples each having 31 columns i.e. 30 input features and 1 output which is the `Class` column
* Print the first 5 data rows

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        data_file = os.path.join(dirname, filename)

# Read data and print the dataframe head
data = pd.read_csv(data_file)
print("Data dimension: " + str(data.shape))
data.head()

Next, let's check if the data is complete

In [ ]:
# Finding any missing value treatment in the dataset.
data_na = data.isna().any()

num_na = (data_na == True).sum()
if num_na > 0:
    data_na_positions = np.where(data_na == True)
    print("Data unavailable for the following columns:")
    print(data_na_positions)
else:
    print("No missing data")

Normalise and drop unwanted features

In [ ]:
data['normalizedAmount'] = StandardScaler().fit_transform(data['Amount'].values.reshape(-1,1))
data = data.drop(['Amount'],axis=1)
data = data.drop(['Time'],axis=1)

print("Amount column has been normalised under the name of NormalizedAmount")
print("Time column has been dropped")

Let's now check the correlation between the different input features with the `Class` column i.e. with the output feature.   
This is important in order to determine which inputs are more important than others.

In [ ]:
# Correlation
data.corrwith(data.Class).plot.bar(
        figsize = (20, 10), title = "Correlation with class", fontsize = 15,
        rot = 45, grid = True)

It shows that `V2`, `V4` and `V11` are the most correlated with `Class` whereas `V17` is the least    correlated as it has a negative correlation.   
Let's double check this by plotting the heatmap

In [ ]:
# HeatMap

# Include all the columns that you do not want them to be included in the heatmap
columns_not_to_include = [] #["Amount", "Time"]

if len(columns_not_to_include) > 0:
    corr = data.iloc[:, np.all([data.columns != c for c in columns_not_to_include], axis=0)].corr()
else:
    corr = data.corr()
# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True
# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(18, 15))
# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

In this section, we will create the training and testing sets by splitting the data into **70%, 30%** split.
Change this value by changing the `test_size` parameter's value in the `train_test_split` function

You can also remove features that are not very well correlated with `Class`.   
To do so, change the `correlation_threshold` parameter's value in `get_features_to_keep`

In [ ]:
# Change the second arguments if you want to minimise the input data by keeping only features that have a correlation with Class that is greater than 0.2 for example
list_of_features = get_features_to_keep(data, correlation_threshold=None)

if len(list_of_features) > 0:
  reduced_data = data.iloc[:, np.any([data.columns == c for c in list_of_features], axis=0)]
  print("Input size reduced from %d to %d" % (data.shape[1] - 1, reduced_data.shape[1] - 1))
else:
  reduced_data = data
  print("Input size remains the same: %d" % (reduced_data.shape[1] - 1))


X = reduced_data.iloc[:, reduced_data.columns != 'Class']
y = reduced_data.iloc[:, reduced_data.columns == 'Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=0)

print("Train dataset has %d samples, %d input features and %d output features" % (X_train.shape[0], X_train.shape[1], y_train.shape[1]))
print("Test dataset has %d samples, %d input features and %d output features" % (X_test.shape[0], X_test.shape[1], y_test.shape[1]))

# Models

## Decision Tree
The first model that we will create is a `Decision Tree` one that has the following characteristics:
* `max_depth=10`
* `max_leaf_nodes=10`

The rest will be set to the default values as described in [Decision Tree Classifier Manual](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

### Training

In [ ]:
%%time
decision_tree = DecisionTreeClassifier(random_state=0,
                                       criterion='gini',
                                       max_depth=10,
                                       max_leaf_nodes=10)
decision_tree.fit(X_train, y_train)

### Prediction

In [ ]:
# Predicting Test Set
y_pred      = decision_tree.predict(X_test)
y_pred_prob = decision_tree.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("Decision Tree Predicion on Test Set:")
print("------------------------------------")
print("Accuracy:\t%.4f" % (accuracy))
print("Precision:\t%.4f" % (precision))
print("Recall:\t\t%.4f" % (recall))
print("F1 Score:\t%.4f" % (f1))

result_dict = {
    'Model': 'Decision Tree',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1
}

try:
    results = results.append(result_dict, ignore_index=True)
except NameError:
    results = pd.DataFrame([['Decision Tree', accuracy, precision, recall, f1, 0]],
               columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score','ROC AUC'])

### Analysis

#### Precision & Recall Curves
Let's first look at the precision and recall curves and see where they intersect, what values correspond to the threshold taken and see if any other threshold value gives a better result for precision and recall.
Remember that:
* `Recall` is the ratio of the detected positives from all the available or true positives
* `Precision` is the ratio of the true positives out of the predicted ones   

In other words, if precision is equal 0.72 and recall is 0.76 it means that a model can detect 76% of the fraud transactions and when it claims that a transaction is fraud, it is correct only 72% of the times.   
One would like both to be high, unfortunately when precision increases, recall decreases. So a compromise needs to be done for each specific application.   
In the case of credit card fraud detection, a high recall is desirable in order to detect all fraud transactions but that means that the precision is low hence many genuine transactions will be flagged as fraud. This is very embarrasing since the credit card company will get heaps of complaints from people whose transaction got rejected.   
Let's now see how the graphs look like, what threshold value was taken and if any other value would be relatively better.

In [ ]:
precisions, recalls, ths = precision_recall_curve(y_test, y_pred_prob[:,1])

# Get the middle threshold, the closest to 0.5 which is used in the classification
th_idx = np.abs(ths - 0.5).argmin()

#plt.figure(figsize=(8,5))

# Plotting the precision, recall curves
plt.plot(ths, precisions[:-1], "b--", label="Precision")
plt.plot(ths, recalls[:-1], "g--", label="Recall")

# Plotting a vertical line at threshold = 0.5 which is the one used to predict
plt.axvline(x=ths[th_idx], color='r', linestyle='--')

# Plotting the intersection of the precision and recall curves
#idx = np.argwhere(np.diff(np.sign(precisions - recalls))).flatten()
idx = get_intersections(precisions, recalls)
plt.plot(ths[idx], recalls[idx], 'ro')

for i in range(len(idx)):
    if i > 0:
        if ((idx[i] - idx[i-1]) / idx[i]) > 0.01:
            plt.annotate('thr = %.1f, prec, rec = %.4f' % (ths[idx[i]], recalls[idx[i]]),
                         xy=(ths[idx[i]], precisions[idx[i]]),
                         xytext=(-40, -50),
                         textcoords='offset points',
                         arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
    else:
        plt.annotate('thr = %.1f, prec, rec = %.4f' % (ths[idx[i]], recalls[idx[i]]),
                     xy=(ths[idx[i]], precisions[idx[i]]),
                     xytext=(-40, -50),
                     textcoords='offset points',
                     arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

plt.plot(ths[th_idx], precisions[th_idx], 'ro')
plt.annotate('precision = %.4f' % (precisions[th_idx]),
             xy=(ths[th_idx], precisions[th_idx]),
             xytext=(10, 20),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
plt.plot(ths[th_idx], recalls[th_idx], 'ro')
plt.annotate('recall = %.4f' % (recalls[th_idx]),
             xy=(ths[th_idx], recalls[th_idx]),
             xytext=(0, -30),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

plt.xlabel('Threshold')
plt.title('Precision / Recall Curves')
plt.legend()

plt.show()

Here we can see that the threshold being used is just above 0.6.   
This gave a recall of 78% and a precision of 87%   
If we look at the intersection of the 2 curves, we can see that for a threshold of 0.2, recall increases by just 1% while precision decreases by 8% which I believe is too much for this small increase in recall. Our system can detect an additional 1% of fraud traffic but 8% increase in people screaming at us!!

In [ ]:
cm    = confusion_matrix(y_test, y_pred) # rows = truth, cols = prediction
df_cm = pd.DataFrame(cm, index = (0, 1), columns = (0, 1))

sns.heatmap(df_cm, annot=True, fmt='g')

The confusion matrix contains 4 sections with rows representing the true values and columns representing the predicted values.
* ** True Negatives *(Top left)*:** Number of samples that are not fraud which the model correctly predicted as not fraud 
* ** True Positives *(Bottom right)*:** Number of samples that are fraud which the model correctly predicted as fraud
* ** False Negatives *(Bottom Left)*:** Number of samples that are fraud which the model predicted as not fraud i.e. this is the number of transactions that were able to fool the model
* ** False Positives *(Top right)*:** Number of samples that are not fraud which the model predicted as fraud i.e. this is the number of transactions that the model mistankenly tried to stop

In [ ]:
dt_fpr, dt_tpr, thresholds = roc_curve(y_test, y_pred_prob[:,1])
roc_auc = auc(dt_fpr, dt_tpr)

results.iloc[-1, results.columns.get_loc('ROC AUC')] = roc_auc

plt.title('Desicion Tree ROC Curve')

plt.plot(dt_fpr, dt_tpr, label='AUC = %0.4f'% roc_auc)
plt.legend(loc='lower right')

plt.plot([0,1],[0,1],'r--')
plt.xlim([-0.001, 1])
plt.ylim([0, 1.001])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

plt.show()

We can see that our curve is not bad with an area under the curve `AUC=0.9114`

Let's see if we can improve it using a different model

## Random Forest
Our second model is a `Random Forst` one which in fact is an ensemble of Decision Tree models. We will use the following characteristics:
* `max_depth=12`
* `n_estimators=250`
* `max_leaf_nodes=10`

The rest will be set to the default values as described in [Random Forest Classifier Manual](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

Note that `n_estimators=250` means that we are creating an ensemble of 250 Decision Tree classifiers.

### Training

In [ ]:
%%time
random_forest = RandomForestClassifier(random_state=0,
                                       max_depth=12,
                                       n_estimators=250,
                                       max_leaf_nodes=100,
                                       n_jobs=-1)
random_forest.fit(X_train, np.ravel(y_train))

### Prediction

In [ ]:
# Predicting Test Set
y_pred      = random_forest.predict(X_test)
y_pred_prob = random_forest.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("Random Forest Predicion on Test Set:")
print("------------------------------------")
print("Accuracy:\t%.4f" % (accuracy))
print("Precision:\t%.4f" % (precision))
print("Recall:\t\t%.4f" % (recall))
print("F1 Score:\t%.4f" % (f1))

result_dict = {
    'Model': 'Random Forest',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1
}

try:
    results = results.append(result_dict, ignore_index=True)
except NameError:
    results = pd.DataFrame([['Random Forest', accuracy, precision, recall, f1, 0]],
               columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score','ROC AUC'])

### Analysis

In [ ]:
precisions, recalls, ths = precision_recall_curve(y_test, y_pred_prob[:,1])

# Get the middle threshold, the closest to 0.5 which is used in the classification
th_idx = np.abs(ths - 0.5).argmin()

# Plotting the precision, recall curves
plt.plot(ths, precisions[:-1], "b--", label="Precision")
plt.plot(ths, recalls[:-1], "g--", label="Recall")

# Plotting a vertical line at threshold = 0.5 which is the one used to predict
plt.axvline(x=ths[th_idx], color='r', linestyle='--')

# Plotting the intersection of the precision and recall curves
idx = get_intersections(precisions, recalls)
plt.plot(ths[idx], recalls[idx], 'ro')

for i in range(len(idx)):
    if i > 0:
        if ((idx[i] - idx[i-1]) / idx[i]) > 0.01:
            plt.annotate('thr = %.2f, prec, rec = %.4f' % (ths[idx[i]], recalls[idx[i]]),
                         xy=(ths[idx[i]], precisions[idx[i]]),
                         xytext=(0, -50),
                         textcoords='offset points',
                         arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
    else:
        plt.annotate('thr = %.2f, prec, rec = %.4f' % (ths[idx[i]], recalls[idx[i]]),
                     xy=(ths[idx[i]], precisions[idx[i]]),
                     xytext=(-40, -50),
                     textcoords='offset points',
                     arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
        


plt.plot(ths[th_idx], precisions[th_idx], 'ro')
plt.annotate('precision = %.4f' % (precisions[th_idx]),
             xy=(ths[th_idx], precisions[th_idx]),
             xytext=(30, -10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
plt.plot(ths[th_idx], recalls[th_idx], 'ro')
plt.annotate('recall = %.4f' % (recalls[th_idx]),
             xy=(ths[th_idx], recalls[th_idx]),
             xytext=(30, 10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

# Plotting an even better threshold
th_idx = np.abs(ths - 0.39).argmin()
plt.axvline(x=ths[th_idx], color='orange', linestyle='--')

plt.plot(ths[th_idx], precisions[th_idx], 'ro')
plt.annotate('%.4f' % (precisions[th_idx]),
             xy=(ths[th_idx], precisions[th_idx]),
             xytext=(-70, 10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
plt.plot(ths[th_idx], recalls[th_idx], 'ro')
plt.annotate('%.4f' % (recalls[th_idx]),
             xy=(ths[th_idx], recalls[th_idx]),
             xytext=(-50, -10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

plt.xlabel('Threshold')
plt.title('Precision / Recall Curves')
plt.legend()

plt.show()

The threshold used is just above 0.5.   
This gave a recall of 77% and a precision of 94%   
If we look at the intersection of the 2 curves, we can see that for a threshold of 0.15, recall increases by 7% while precision decreases by 11% which I believe is too much for this small recall increase. Our system can detect an additional 7% of fraud traffic but 11% increase in people screaming at us!!   
On the other hand, if we look at the orange vertical line at threshold of 0.39, we can see that we get a recall of 80% (a 3% increase) and a precision of 94.4% (also a slight increase of 0.23%). This is definitely a better choice to consider for our model.

In [ ]:
cm    = confusion_matrix(y_test, y_pred) # rows = truth, cols = prediction
df_cm = pd.DataFrame(cm, index = (0, 1), columns = (0, 1))

sns.heatmap(df_cm, annot=True, fmt='g')

In [ ]:
rf_fpr, rf_tpr, thresholds = roc_curve(y_test, y_pred_prob[:,1])
roc_auc = auc(rf_fpr, rf_tpr)

results.iloc[-1, results.columns.get_loc('ROC AUC')] = roc_auc

plt.title('Random Forest ROC Curve')

plt.plot(rf_fpr, rf_tpr, label='AUC = %0.4f'% roc_auc)
plt.legend(loc='lower right')

plt.plot([0,1],[0,1],'r--')
plt.xlim([-0.001, 1])
plt.ylim([0, 1.001])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

plt.show()

From `AUC=0.9114` for the `Decision Tree`, we have improved to `AUC=0.9762` for an ensemble of them: a `Random Forest` model.

Let's see if we can improve it using a neural network model

## TensorFlow Keras Model

In [ ]:
# Create model
nn_model = Sequential()

nn_model.add(Dense(16, input_dim = X_train.shape[1]))
nn_model.add(BatchNormalization())
nn_model.add(Activation('relu'))
nn_model.add(Dense(12))
nn_model.add(BatchNormalization())
nn_model.add(Activation('relu'))
nn_model.add(Dense(1, activation = 'sigmoid'))

nn_model.compile(optimizer=Adam(lr=1e-3) , loss='binary_crossentropy', metrics = [tf.keras.metrics.AUC()])

# Model Summary
nn_model.summary()

### Training

In [ ]:
%%time

epochs=20
batch_size=512

h = nn_model.fit(X_train, y_train,
                 batch_size=batch_size,
                 epochs=epochs)

### Prediction

In [ ]:
y_pred_prob = nn_model.predict(X_test)
y_pred      = (y_pred_prob > 0.5)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("Neural Network Predicion on Test Set:")
print("-------------------------------------")
print("Accuracy:\t%.4f" % (accuracy))
print("Precision:\t%.4f" % (precision))
print("Recall:\t\t%.4f" % (recall))
print("F1 Score:\t%.4f" % (f1))

result_dict = {
    'Model': 'Keras NN',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1
}

try:
    results = results.append(result_dict, ignore_index=True)
except NameError:
    results = pd.DataFrame([[' Neural Networks', accuracy, precision, recall, f1, 0]],
               columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'])

### Analysis

In [ ]:
precisions, recalls, ths = precision_recall_curve(y_test, y_pred_prob)

# Get the middle threshold, the closest to 0.5 which is used in the classification
th_idx = np.abs(ths - 0.5).argmin()

# Plotting the precision, recall curves
plt.plot(ths, precisions[:-1], "b--", label="Precision")
plt.plot(ths, recalls[:-1], "g--", label="Recall")

# Plotting a vertical line at threshold = 0.5 which is the one used to predict
plt.axvline(x=ths[th_idx], color='r', linestyle='--')

# Plotting the intersection of the precision and recall curves
idx = get_intersections(precisions, recalls)
plt.plot(ths[idx], recalls[idx], 'ro')

for i in range(len(idx)):
    if i > 0:
        if ((idx[i] - idx[i-1]) / idx[i]) > 0.01:
            plt.annotate('thr = %1.f, prec, rec = %.4f' % (ths[idx[i]], recalls[idx[i]]),
                         xy=(ths[idx[i]], precisions[idx[i]]),
                         xytext=(0, -50),
                         textcoords='offset points',
                         arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
    else:
        plt.annotate('thr = %.1f, prec, rec = %.4f' % (ths[idx[i]], recalls[idx[i]]),
                     xy=(ths[idx[i]], precisions[idx[i]]),
                     xytext=(-20, -50),
                     textcoords='offset points',
                     arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
        


plt.plot(ths[th_idx], precisions[th_idx], 'ro')
plt.annotate('precision = %.4f' % (precisions[th_idx]),
             xy=(ths[th_idx], precisions[th_idx]),
             xytext=(15, 20),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
plt.plot(ths[th_idx], recalls[th_idx], 'ro')
plt.annotate('recall = %.4f' % (recalls[th_idx]),
             xy=(ths[th_idx], recalls[th_idx]),
             xytext=(30, 0),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

# Plotting an even better threshold
th_idx = np.abs(ths - 0.42).argmin()
plt.axvline(x=ths[th_idx], color='orange', linestyle='--')

plt.plot(ths[th_idx], precisions[th_idx], 'ro')
plt.annotate('%.4f' % (precisions[th_idx]),
             xy=(ths[th_idx], precisions[th_idx]),
             xytext=(-70, 10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
plt.plot(ths[th_idx], recalls[th_idx], 'ro')
plt.annotate('%.4f' % (recalls[th_idx]),
             xy=(ths[th_idx], recalls[th_idx]),
             xytext=(-50, -10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))


plt.xlabel('Threshold')
plt.title('Precision / Recall Curves')
plt.legend()

plt.show()

Here we can see that the threshold that was used is 0.5.   
This gave a recall of 74% and a precision of 87%   
If we look at the intersection of the 2 curves, we can see that for a threshold of 0.2, recall increases by about 6% while precision decreases by almost 6%   
If we look at the orange vertical line at threshold of 0.42, we can see that we get a recall of 77% (a 3% increase) and a precision of 85% (a 2% decrease ... it's even 1.5% decrease).

In [ ]:
cm    = confusion_matrix(y_test, y_pred) # rows = truth, cols = prediction
df_cm = pd.DataFrame(cm, index = (0, 1), columns = (0, 1))

sns.heatmap(df_cm, annot=True, fmt='g')

In [ ]:
nn_fpr, nn_tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(nn_fpr, nn_tpr)

results.iloc[-1, results.columns.get_loc('ROC AUC')] = roc_auc

plt.title('Keras Model ROC Curve')

plt.plot(nn_fpr, nn_tpr, label='AUC = %0.4f'% roc_auc)
plt.legend(loc='lower right')

plt.plot([0,1],[0,1],'r--')
plt.xlim([-0.001, 1])
plt.ylim([0, 1.001])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

plt.show()

From `AUC=0.9114` for the `Decision Tree`, to `AUC=0.9762` a `Random Forest` model, we slightly improved to `AUC=0.9772` for a TensorFlow Keras network

# Conclusion
Let's print all the results as well as plot all ROC curves

In [ ]:
print(results)

In [ ]:
plt.title('ROC Curve - Model Comparison')

plt.plot(dt_fpr, dt_tpr, label='Decision Tree')
plt.plot(rf_fpr, rf_tpr, label='Random Forest')
plt.plot(nn_fpr, nn_tpr, label='Keras')
plt.legend(loc='lower right')

plt.plot([0,1],[0,1],'r--')
plt.xlim([-0.001, 1])
plt.ylim([0, 1.001])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

plt.show()

The TF Keras NN gave a result almost identical to the Random Forest one however training time is much faster.

## What to do next?
Feel free to try changing the parameters in order to get better performances for these models.   
You can try the following:
* Change the parameters for the 3 different models in order to get better or bigger models (more units, more layers, different max_depth, ...)
* What happens when amount is not normalised? Or when V1, ... V28 are normalised?
* What happens if Time is not dropped?
* What if the threshold is decreased or increased? Are there better options to get better performance parameters? Especially if the models have been changed and the recall / precision analysis is no longer valid.

**All comments are welcome!!**   
Feel free to upvote if you find this kernel helpful.   
Thank you!!

# Acknowledgement
* This work has been highly inspired by the work done **Aniruddha Choudhury** [found here](https://blog.usejournal.com/credit-card-fraud-detection-by-neural-network-in-keras-4bd81cc9e7fe)
* A special thanks to **Aurélien Géron** for his book `Hands-on Machine Learning with Scikit-Learn, Keras & Tensorflow (Second Edition)` which has not been released yet (August 2019) but I had the priviledge to check ... [Check it out](https://www.amazon.com/Hands-Machine-Learning-Scikit-Learn-TensorFlow/dp/1492032646/ref=sr_1_1?crid=2T8QXHIGNL9LL&keywords=aurelien+geron&qid=1566807558&s=gateway&sprefix=Geron+au%2Caps%2C250&sr=8-1)